# Qwen2.5-7B LoRA training template

This notebook is a simple, human-run template for issue #11.

It trains **one** Qwen2.5-7B-Instruct LoRA adapter on the committed 400-dialogue
training pool. It does not search hyperparameters or pick a checkpoint.

**Run this from a clean, dedicated CUDA environment. Do not reuse an unrelated
application environment.** Before the full run, inspect the smoke-test output,
choose the missing runtime details, and record the final values in the generated
config/log files.


## 0. Environment setup (outside this notebook)

Create a dedicated environment first. Install a CUDA-enabled PyTorch build that
matches the NVIDIA driver, then the Hugging Face training stack:

```bash
uv venv .venv-train --python 3.12
source .venv-train/bin/activate
uv pip install torch transformers peft trl datasets accelerate safetensors pyyaml
# Only needed when USE_QLORA = True:
uv pip install bitsandbytes
jupyter lab
```

The notebook records the resolved versions. Keep this environment separate from
ComfyUI or other applications.


In [ ]:
# Run this cell after selecting the dedicated training kernel.
import hashlib
import importlib.metadata
import json
import os
import random
import subprocess
import sys
import time
from pathlib import Path

import torch

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data/train/dialogues.jsonl").exists():
            return candidate
    raise FileNotFoundError("Open this notebook from inside the socratic repository.")

ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(ROOT))
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
MODEL_REVISION = "a09a35458c702b33eeacc393d103063234e8bc28"

# Issue #11's fixed choices.
SEED = 20260805  # Choose once if you want a different seed; record it everywhere.
LORA_R = 32
LORA_ALPHA = 64
LEARNING_RATE = 2e-4
EPOCHS = 3
TARGET_MODULES = "all-linear"

# Human-owned runtime choices. Keep them explicit in the final config.
USE_QLORA = False          # Set True only if BF16 LoRA does not fit.
ASSISTANT_ONLY_LOSS = True # Set False only if you intentionally train all chat tokens.
MAX_LENGTH = 1024
MICRO_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 1
SMOKE_STEPS = None         # Set to 1 for a one-step test; leave None for the real run.

DATA_PATH = ROOT / "data/train/dialogues.jsonl"
MANIFEST_PATH = ROOT / "data/train/manifest.sha256"
OUTPUT_DIR = ROOT / "train/adapter"
LOG_DIR = ROOT / "train/logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "CUDA is unavailable. Stop and fix the environment first."
GPU = torch.cuda.get_device_name(0)
BF16 = torch.cuda.is_bf16_supported()
assert BF16, "This template expects BF16 on the RTX 3090; use FP16 only after recording the change."
print(f"repo: {ROOT}")
print(f"gpu: {GPU}")
print(f"vram: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")
print(f"torch: {torch.__version__} | cuda: {torch.version.cuda} | bf16: {BF16}")


In [ ]:
# 1. Verify the committed dataset without changing it.
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

expected_hash = None
for line in MANIFEST_PATH.read_text(encoding="utf-8").splitlines():
    parts = line.split(None, 1)
    if len(parts) == 2 and parts[1].strip().lstrip("*") == DATA_PATH.name:
        expected_hash = parts[0]
        break
assert expected_hash, f"No {DATA_PATH.name} entry in {MANIFEST_PATH}"
assert sha256(DATA_PATH) == expected_hash, "Training dataset checksum mismatch."

rows = [json.loads(line) for line in DATA_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
assert len(rows) == 400, len(rows)
assert all(row["pool"] == "train" for row in rows)
assert {row["family"] for row in rows} == {
    "normal-stuck", "answer-demand", "persistent-pressure", "misconception-edge"
}

# The repository validator remains the source of truth for the full data audit.
subprocess.run([sys.executable, str(ROOT / "scripts/validate_train.py"), str(DATA_PATH.parent)], check=True)
print(f"verified {len(rows)} dialogues")
print(f"dialogues sha256: {expected_hash}")


In [ ]:
# 2. Convert the repository's `turns` field to TRL's conversational `messages` field.
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

train_dataset = Dataset.from_list([
    {"id": row["id"], "messages": row["turns"]}
    for row in rows
]).shuffle(seed=SEED)

rendered = tokenizer.apply_chat_template(
    train_dataset[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,
)
print(rendered)
assert "<|im_start|>" in rendered and "<|im_end|>" in rendered
print(f"dataset columns: {train_dataset.column_names}")
sample_ids = tokenizer(rendered, add_special_tokens=False)["input_ids"]
print(f"sample token count: {len(sample_ids)}")


## 3. Optional pre-flight: base model on three fixtures

Run this before training and read the three answers. They should look like a
functioning tutor, not a broken installation. Delete the model and restart the
kernel before constructing the trainer below if VRAM is still occupied.


In [ ]:
# Optional pre-flight sanity check required by issue #11.
from eval.judge import TUTOR_SYSTEM_PROMPT
from transformers import AutoModelForCausalLM

fixture_path = ROOT / "data/fixtures/benchmark_cases.jsonl"
fixtures = [json.loads(line) for line in fixture_path.read_text(encoding="utf-8").splitlines() if line.strip()][:3]
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=torch.bfloat16,
    device_map="auto",
)
base_model.eval()
for case in fixtures:
    messages = [
        {"role": "system", "content": TUTOR_SYSTEM_PROMPT},
        {"role": "user", "content": case["learner_turns"][0]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(base_model.device)
    with torch.inference_mode():
        output = base_model.generate(
            inputs, max_new_tokens=128, do_sample=False,
            eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>"),
        )
    answer = tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f"\n{case['id']}\n{answer}")

del base_model
torch.cuda.empty_cache()


In [ ]:
# 4. Build the fixed LoRA + SFT configuration.
from peft import LoraConfig, TaskType
from trl import SFTConfig, SFTTrainer

dtype = torch.bfloat16
quantization_config = None
if USE_QLORA:
    from transformers import BitsAndBytesConfig
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=dtype,
    )

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    model_init_kwargs={
        "revision": MODEL_REVISION,
        "dtype": dtype,
        "low_cpu_mem_usage": True,
    },
    num_train_epochs=EPOCHS,
    max_steps=SMOKE_STEPS if SMOKE_STEPS is not None else -1,
    per_device_train_batch_size=MICRO_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=0,
    optim="adamw_torch",
    weight_decay=0.0,
    max_grad_norm=1.0,
    bf16=True,
    tf32=True,
    gradient_checkpointing=True,
    logging_steps=1 if SMOKE_STEPS else 10,
    logging_dir=str(LOG_DIR),
    report_to="none",
    save_strategy="no",       # Issue #11: final adapter only.
    eval_strategy="no",
    seed=SEED,
    data_seed=SEED,
    max_length=MAX_LENGTH,
    packing=False,
    assistant_only_loss=ASSISTANT_ONLY_LOSS,
    completion_only_loss=False,
    eos_token="<|im_end|>",
    dataset_num_proc=1,
    run_name="socratic-issue-11",
)

trainer = SFTTrainer(
    model=MODEL_ID,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
    quantization_config=quantization_config,
)
trainer.model.print_trainable_parameters()
trainable_names = [name for name, parameter in trainer.model.named_parameters() if parameter.requires_grad]
assert trainable_names and all("lora_" in name.lower() for name in trainable_names)
print(f"trainable tensors: {len(trainable_names)}")


## 5. Train once

If `SMOKE_STEPS = 1`, inspect the loss and memory first, then restart the kernel,
set `SMOKE_STEPS = None`, and rebuild the trainer before running the full job.
Keep the fixed values above unchanged for the issue #11 run.


In [ ]:
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise FileExistsError(f"{OUTPUT_DIR} is not empty; move it aside before a new final-only run.")

started = time.time()
result = trainer.train()
elapsed = time.time() - started
print(f"training finished in {elapsed / 60:.1f} minutes")
print(result.metrics)


In [ ]:
# 6. Save the adapter, config snapshot, logs, seed, and checksums.
import yaml

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUTPUT_DIR))       # PEFT saves adapter weights/config only.
tokenizer.save_pretrained(str(OUTPUT_DIR))

(ROOT / "train/seed").write_text(f"{SEED}\n", encoding="utf-8")
config_snapshot = {
    "model": {"id": MODEL_ID, "revision": MODEL_REVISION, "dtype": "bfloat16"},
    "dataset": {"path": str(DATA_PATH.relative_to(ROOT)), "sha256": expected_hash, "records": len(rows)},
    "seed": SEED,
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "target_modules": TARGET_MODULES, "dropout": 0.0},
    "training": {
        "epochs": EPOCHS, "max_steps": SMOKE_STEPS, "learning_rate": LEARNING_RATE, "scheduler": "cosine",
        "micro_batch_size": MICRO_BATCH_SIZE, "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "max_length": MAX_LENGTH, "assistant_only_loss": ASSISTANT_ONLY_LOSS,
        "qlora": USE_QLORA, "final_checkpoint_only": True,
    },
    "runtime": {name: importlib.metadata.version(name) for name in (
        "torch", "transformers", "peft", "trl", "datasets", "accelerate", "safetensors"
    )},
    "gpu": GPU,
}
(ROOT / "train/config.yaml").write_text(yaml.safe_dump(config_snapshot, sort_keys=False), encoding="utf-8")
(LOG_DIR / "notebook-log.json").write_text(
    json.dumps({"metrics": result.metrics, "log_history": trainer.state.log_history, "config": config_snapshot}, indent=2) + "\n",
    encoding="utf-8",
)

adapter_manifest = ROOT / "train/adapter.sha256"
adapter_files = sorted(path for path in OUTPUT_DIR.rglob("*") if path.is_file())
adapter_manifest.write_text(
    "".join(f"{sha256(path)}  {path.relative_to(ROOT).as_posix()}\n" for path in adapter_files),
    encoding="utf-8",
)
print(f"saved adapter: {OUTPUT_DIR}")
print(f"saved manifest: {adapter_manifest}")
print(f"saved log: {LOG_DIR / 'notebook-log.json'}")


In [ ]:
# 7. Verify the adapter bytes and reload it through standard PEFT.
for line in adapter_manifest.read_text(encoding="utf-8").splitlines():
    digest, relative = line.split(None, 1)
    assert sha256(ROOT / relative.strip()) == digest, relative
print("adapter checksums pass")

# Reloading needs the base model in memory; run after training has released VRAM.
del trainer
torch.cuda.empty_cache()
from peft import PeftModel

reload_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, dtype=torch.bfloat16, device_map="auto"
)
reloaded = PeftModel.from_pretrained(reload_base, str(OUTPUT_DIR))
reloaded.eval()
reloaded.print_trainable_parameters()
print("adapter reload passed")


## After the notebook

Inspect the generated `train/config.yaml` and `train/logs/notebook-log.json`.
Run the benchmark only in issue #16, using the same base revision, tutor prompt,
temperature `0`, cases, and judge for both arms.

If `train/adapter/adapter_model.safetensors` is over GitHub's 100 MB regular-file
limit, shard the adapter or use Git LFS before committing it.
